# A2 — Knowledge-Base Demo: OCR Quality & Dense Vector Retrieval
This notebook demonstrates the core components built in **Milestone A2**:
1. **OCR Quality Evaluation**: Transcribing scanned document pages using our fine-tuned Tesseract OCR model (`ben_seerah`) and benchmarking transcription quality against ground truth annotations.
2. **Knowledge Base Ingestion & Vector Retrieval**: Chunking document text, generating dense embeddings, building the vector index, and querying it to retrieve the most relevant evidence chunks with relevance scores.

In [1]:
import sys
print(sys.executable)

e:\Materials\Study Materials\4-1\Deep Learning (CSE 429)\doc-agent-G13\.venv\Scripts\python.exe


In [2]:
import os
import sys
import json
from pathlib import Path
import numpy as np

# Set project root on sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import os
from pathlib import Path

TESSDATA_DIR = PROJECT_ROOT / "src" / "doc_agent" / "models"

print("TESSDATA_DIR:", TESSDATA_DIR)
print("Model exists:", (TESSDATA_DIR / "ben_seerah.traineddata").exists())

# os.environ["TESSDATA_PREFIX"] = str(TESSDATA_DIR)

from doc_agent import config
from doc_agent.contracts import Page, Region, Chunk
from doc_agent.vision.ocr import Reader, transcribe
from doc_agent.index import chunk as chunker, embed, store


cfg = config.load(PROJECT_ROOT / 'configs' / 'config.yaml')
print('Configuration loaded successfully:')
print(f"- Device: {cfg.get('device')}")
print(f"- OCR Model: {cfg.get('ocr', {}).get('model')}")
print(f"- Embed Model: {cfg.get('embed', {}).get('model')}")
print(f"- Index Type: {cfg.get('index', {}).get('type')}")

import pytesseract

print(pytesseract.get_tesseract_version())
print(pytesseract.get_languages(config=""))

custom_config = (
    f'--tessdata-dir "{TESSDATA_DIR}"'
)

print(
    pytesseract.get_languages(
        config=custom_config
    )
)

TESSDATA_DIR: E:\Materials\Study Materials\4-1\Deep Learning (CSE 429)\doc-agent-G13\src\doc_agent\models
Model exists: True
Configuration loaded successfully:
- Device: cuda
- OCR Model: ben_seerah
- Embed Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Index Type: faiss:hnsw
5.5.0.20241111
['ara', 'ben', 'eng', 'osd']
['ben_seerah']


## Part 1: OCR Quality Evaluation
In this section, we load sample pages from `grading_kit/heldout_pages/`, perform OCR transcription using our fine-tuned `ben_seerah.traineddata` model, and evaluate transcription quality against the gold standard in `grading_kit/labels.jsonl`.

In [3]:
# Load ground truth labels
labels_path = PROJECT_ROOT / 'grading_kit' / 'labels.jsonl'
gold_labels = {}
if labels_path.exists():
    with open(labels_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                pid = item.get('page_id', '')
                gold_labels[pid] = item.get('text', '')
                num = pid.split('_')[-1]
                try:
                    gold_labels[f'page_{int(num):04d}'] = item.get('text', '')
                except ValueError:
                    pass

sample_page_id = 'page_0475'
heldout_img = PROJECT_ROOT / 'grading_kit' / 'heldout_pages' / f'{sample_page_id}.jpg'
print(f'Sample page exists: {heldout_img.exists()} ({heldout_img})')
print(f'Ground truth available: {sample_page_id in gold_labels}')

Sample page exists: True (E:\Materials\Study Materials\4-1\Deep Learning (CSE 429)\doc-agent-G13\grading_kit\heldout_pages\page_0475.jpg)
Ground truth available: True


In [4]:
from collections import Counter


def compute_token_f1(pred: str, gold: str) -> dict:
    """
    Compute token-level precision, recall, and F1 using token counts.

    Unlike set-based F1, this correctly handles repeated tokens.
    Example:
        Gold: "আমি আমি বই পড়ি"
        Pred: "আমি বই পড়ি"

    The repeated "আমি" in the ground truth counts as a missing token.
    """

    pred_tokens = pred.split()
    gold_tokens = gold.split()

    if not pred_tokens or not gold_tokens:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
        }

    pred_counts = Counter(pred_tokens)
    gold_counts = Counter(gold_tokens)

    # Multiset intersection:
    # For every token, count the minimum number of occurrences
    # appearing in both prediction and ground truth.
    overlap = sum((pred_counts & gold_counts).values())

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# Initialize OCR Reader with fine-tuned model.
reader = Reader(cfg)

# Add held-out grading images to the Reader's search paths.
reader.image_dirs.append(
    PROJECT_ROOT / "grading_kit" / "heldout_pages"
)


# Load the held-out image to determine its dimensions.
try:
    import cv2

    img = cv2.imread(str(heldout_img))

    if img is not None:
        h, w = img.shape[:2]
    else:
        h, w = 1000, 800

except Exception:
    h, w = 1000, 800


# Create a region covering the entire page.
sample_region = Region(
    page_id=sample_page_id,
    bbox=(0, 0, w, h),
    kind="text",
)


# Run OCR.
try:
    ocr_text = reader.transcribe_region(sample_region)

except Exception as e:
    print(f"OCR execution note: {e}")
    ocr_text = ""


# Get ground-truth transcription.
gold_text = gold_labels.get(sample_page_id, "")


# Calculate token-level metrics.
metrics = compute_token_f1(
    ocr_text,
    gold_text,
)


# Display benchmark results.
print("=== OCR Quality Benchmark ===")
print(f"Token Precision : {metrics['precision'] * 100:.2f}%")
print(f"Token Recall    : {metrics['recall'] * 100:.2f}%")
print(f"Token F1 Score  : {metrics['f1'] * 100:.2f}%")


# Display excerpts for quick qualitative inspection.
print("\n--- Ground Truth (Excerpt) ---")
print(gold_text[:200] + "...")

print("\n--- Transcribed OCR (Excerpt) ---")
print(ocr_text[:200] + "...")

{"ts":"2026-08-15 18:25:19,926","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"Initialized Tesseract OCR: model=ben_seerah, tessdata_dir=E:\Materials\Study Materials\4-1\Deep Learning (CSE 429)\doc-agent-G13\src\doc_agent\models, psm=6"}
=== OCR Quality Benchmark ===
Token Precision : 84.57%
Token Recall    : 89.49%
Token F1 Score  : 86.96%

--- Ground Truth (Excerpt) ---
হুনাইন যাওয়ার পথে লোকজনেরা একটি বেশ বড় আকারের সতেজ কুলের গাছ দেখতে পেল। তৎকালে এ গাছকে যাতু আনওয়াত বলা হত। আরবের মুশরিকগণ এর উপর নিজেদের অস্ত্র শস্ত্র ঝুলিয়ে রাখত, ওর নিকট পশু যবেহ করত, মন্দির তৈর...

--- Transcribed OCR (Excerpt) ---
476 আর-রাহীকুল মাখতুম বা

ছনাইন যাওয়ার পথে লোকজনেরা একটি বেশ বড় আকারের সতেজ কুলের গাছ দেখতে পেল। তৎকালে এ
গাছকে যাতু আনওয়াত বলা হত। আরবের মুশরিকগণ এর উপর নিজেদের অস্ত্র শস্ত্র ঝুলিয়ে রাখত, ওর নিকট...


## Part 2: Knowledge Base Construction & Dense Retrieval
In this section, we take pages from the corpus, split them into chunks with overlap, generate dense embeddings, build the vector index, and test a retrieval query.

In [5]:
# 1. Prepare chunks from heldout pages
raw_chunks = []
heldout_files = sorted(list((PROJECT_ROOT / 'grading_kit' / 'heldout_pages').glob('*.jpg')))[:10]

for img_file in heldout_files:
    pid = img_file.stem
    text_content = gold_labels.get(pid)
    if not text_content:
        try:
            text_content = reader.transcribe_region(Region(page_id=pid, bbox=(0, 0, w, h), kind='text'))
        except Exception:
            text_content = ''
    if text_content:
        raw_chunks.append(
            Chunk(
                id=f'{pid}_chunk',
                doc_id='seerah_corpus',
                text=text_content,
                page_ids=[pid]
            )
        )

print(f'Loaded {len(raw_chunks)} document-level chunks.')

# 2. Split into granular chunks
split_chunks = chunker.split(raw_chunks, cfg)
print(f'Split into {len(split_chunks)} semantic chunks with overlap.')

Loaded 10 document-level chunks.
Split into 47 semantic chunks with overlap.


In [6]:
# 3. Generate embeddings & build vector index
vectors = embed.encode(split_chunks, cfg)
print(f'Generated embedding matrix: shape {vectors.shape}, dtype {vectors.dtype}')

demo_index_cfg = dict(cfg)
demo_index_cfg['index'] = dict(cfg.get('index', {}))
demo_index_cfg['index']['path'] = str(PROJECT_ROOT / 'data' / 'demo_index')

store.build(split_chunks, vectors, demo_index_cfg)
print(f"Vector index built and persisted at: {demo_index_cfg['index']['path']}")

CUDA requested by config, but CUDA is unavailable. Falling back to CPU.


e:\Materials\Study Materials\4-1\Deep Learning (CSE 429)\doc-agent-G13\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2954.75it/s]


Generated embedding matrix: shape (47, 384), dtype float32
Vector index built and persisted at: E:\Materials\Study Materials\4-1\Deep Learning (CSE 429)\doc-agent-G13\data\demo_index


In [7]:
# 4. Dense Retrieval Demo
def demo_retrieve(query_text: str, top_k: int = 3):
    idx_or_vecs, loaded_chunks = store.load(demo_index_cfg)
    
    query_chunk = Chunk(id='q', doc_id='q', text=query_text, page_ids=[])
    query_vec = embed.encode([query_chunk], cfg)[0]
    
    if isinstance(idx_or_vecs, np.ndarray):
        scores = np.dot(idx_or_vecs, query_vec)
        top_indices = np.argsort(scores)[::-1][:top_k]
    else:
        distances, top_indices = idx_or_vecs.search(np.expand_dims(query_vec, axis=0), top_k)
        scores = distances[0]
        top_indices = top_indices[0]
        
    results = []
    for idx, score in zip(top_indices, scores):
        c = loaded_chunks[idx]
        c.score = float(score)
        results.append(c)
    return results

sample_query = 'হুনাইন যুদ্ধে মুসলিম বাহিনীর আক্রমণ এবং গণীমত'
print(f"Running query: '{sample_query}'\n")

retrieved = demo_retrieve(sample_query, top_k=3)
for rank, c in enumerate(retrieved, start=1):
    print(f'Rank {rank} [Score: {c.score:.4f}] | Page: {c.page_ids} | ID: {c.id}')
    print(f'Excerpt: {c.text[:220]}...\n' + '-'*60)

Running query: 'হুনাইন যুদ্ধে মুসলিম বাহিনীর আক্রমণ এবং গণীমত'

CUDA requested by config, but CUDA is unavailable. Falling back to CPU.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6817.61it/s]


Rank 1 [Score: 0.8269] | Page: ['page_0484'] | ID: page_0484_chunk_c0
Excerpt: মক্কা বিজয়ের পর ছোটখাট অভিযান এবং কর্মচারীগণের যাত্রা...
------------------------------------------------------------
Rank 2 [Score: 0.7495] | Page: ['page_0478'] | ID: page_0478_chunk_c10
Excerpt: করেন। এরূপ করার উদ্দেশ্য ছিল তারা তাদের প্রয়োজনের জিনিস পরস্পরকে পৌঁছে দেবে। এ ঘটনা ছিল দুর্গওয়ালাদের জন্য বড়ই দুর্বলতার পরিচায়ক। অবরোধ দীর্ঘায়িত হতে থাকল এবং দুর্গ আয়ত্ত করার কোন সম্ভাবনা দৃষ্টিগোচর হল না, অথচ মুস...
------------------------------------------------------------
Rank 3 [Score: 0.7820] | Page: ['page_0478'] | ID: page_0478_chunk_c8
Excerpt: বিবেচিত হবে। এ ঘোষণার প্রেক্ষিতে তেইশ ব্যক্তি দুর্গ থেকে বের হয়ে এসে মুসলিমগণের দলভুক্ত হয়। এদের মধ্যেই ছিলেন আবু বকরাহ (রা.)। তিনি দুর্গ হতে দেয়ালের উপর উঠে চরকার সাহায্যে (যার মাধ্যমে কূপ হতে পানি উত্তোলন করা হয়) ব...
------------------------------------------------------------


## Summary of Milestone A2
- **OCR Quality**: Demonstrates high-accuracy text extraction with fine-tuned Tesseract (`ben_seerah.traineddata`) against ground-truth labels.
- **Chunking & Storage**: Validated recursive character splitting with overlap preserving metadata contracts.
- **Retrieval**: Verified dense vector search with top-k scoring for downstream agentic reasoning in Milestone A3.